<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session12_tool_calling_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۲: فراخوانی ابزار (Tool Calling)

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session12_tool_calling.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/store_tools.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import json

import pandas as pd
from store_tools import STORE_TOOLS, db, place_order, run_tool, tool_schema

SYSTEM = ("You are the assistant of Novin Kala, an Iranian online store. Answer in Persian. "
          "Use the tools for prices, stock, shipping and arithmetic; never guess numbers.")


def run_with_tools(question, tools, max_rounds=6, executor=run_tool):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_rounds):
        message = client.chat.completions.create(model=MODEL, messages=messages,
                                                 tools=[tool_schema(t) for t in tools], temperature=0).choices[0].message
        if not message.tool_calls:
            return message.content
        messages.append(message.model_dump(exclude_none=True))
        for call in message.tool_calls:
            result = executor(call.function.name, call.function.arguments, tools)
            print(f"  -> {call.function.name}({call.function.arguments}): {result[:150]}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "Stopped: too many tool calls."

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

ابزاری به نام `cancel_order(order_id)` بنویسید که طبق سیاست نوین‌کالا فقط سفارش‌هایی را لغو کند که هنوز «ارسال شده» نیستند. آن را با تأیید کاربر به حلقه اضافه کنید و با هر دو سفارش نمونهٔ دیتابیس آزمایش کنید.

</div>

In [ ]:
CANCELLABLE = {"در انتظار پرداخت", "در حال پردازش", "آماده ارسال"}


def cancel_order(order_id: str) -> dict:
    """Cancel an order that has not been shipped yet. This changes data: call it only after the customer confirmed.

    Args:
        order_id: the tracking code, e.g. "NK519027".
    """
    row = db.execute("SELECT status FROM orders WHERE order_id = ?", (order_id.strip().upper(),)).fetchone()
    if row is None:
        raise ValueError(f"no order with code {order_id}")
    if row[0] not in CANCELLABLE:
        raise ValueError(f"an order with status '{row[0]}' cannot be cancelled; use the return process instead")
    db.execute("UPDATE orders SET status = 'لغو شده' WHERE order_id = ?", (order_id.strip().upper(),))
    return {"order_id": order_id.upper(), "status": "لغو شده"}


def confirm_then_run(name, arguments, tools):
    if name in {"cancel_order", "place_order"}:
        if input(f"Run {name}({arguments})? (y/n) ").strip().lower() != "y":
            return json.dumps({"error": "the customer did not confirm"})
    return run_tool(name, arguments, tools)


tools = STORE_TOOLS + [cancel_order]
print(run_tool("cancel_order", '{"order_id": "NK482913"}', tools))     # shipped -> refused by the policy
print(run_with_tools("سفارش NK519027 رو لغو کن.", tools, executor=confirm_then_run))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

قاعدهٔ سیاست («لغو تا پیش از ارسال») در **کد ابزار** پیاده شده است، نه در پرامپت. حتی اگر مدل اشتباه کند و برای سفارش ارسال‌شده لغو بخواهد، ابزار آن را رد می‌کند و پیام خطا می‌گوید راه درست چیست؛ مدل می‌تواند همین را به مشتری بگوید.

نمونهٔ ارسال‌شده (`NK482913`) با خطای سیاست رد می‌شود و نمونهٔ در حال پردازش (`NK519027`) بعد از تأیید لغو می‌شود. اگر تأیید ندهید، مدل پیام «the customer did not confirm» را می‌بیند و باید به مشتری بگوید سفارش لغو نشد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

ده پرسش بنویسید و برای هر کدام ابزار درستی را که باید اول صدا زده شود (یا «هیچ‌کدام») مشخص کنید. درصد انتخاب درست ابزار را اندازه بگیرید.

</div>

In [ ]:
SELECTION_TESTS = [
    ("قیمت گلکسی A55 چنده؟", "search_products"),
    ("مشخصات کامل کالای P2003 رو بگو", "get_product"),
    ("سفارش NK482913 کجاست؟", "track_order"),
    ("ارسال سریع به مشهد چقدر میشه؟", "shipping_cost"),
    ("۲۸۹۰۰۰۰۰ ضربدر ۳ منهای ۱۰ درصد چند میشه؟", "calculator"),
    ("تلویزیون زیر ۲۰ میلیون دارید؟", "search_products"),
    ("هزینه ارسال عادی سفارش ۵۰۰ هزار تومانی به رشت؟", "shipping_cost"),
    ("وضعیت سفارش NK519027", "track_order"),
    ("سلام، خوبی؟", None),
    ("پایتخت ایتالیا کجاست؟", None),
]


def first_tool(question, schemas):
    message = client.chat.completions.create(
        model=MODEL, temperature=0, tools=schemas,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]).choices[0].message
    return message.tool_calls[0].function.name if message.tool_calls else None


def selection_accuracy(schemas):
    rows = [{"question": q, "expected": e, "chosen": first_tool(q, schemas)} for q, e in SELECTION_TESTS]
    df = pd.DataFrame(rows)
    return (df.expected.fillna("-") == df.chosen.fillna("-")).mean(), df


schemas = [tool_schema(t) for t in STORE_TOOLS]
accuracy, details = selection_accuracy(schemas)
print(f"tool selection accuracy: {accuracy:.0%}")
details

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

اشتباه‌های رایج: برای «مشخصات P2003» به جای `get_product`، `search_products` با شناسه صدا زده می‌شود، که نتیجهٔ درستی نمی‌دهد چون جستجو روی نام است. یا برای پرسش ریاضی، مدل خودش حساب می‌کند و ابزار نمی‌زند. هر دو با توضیح دقیق‌تر ابزار بهتر می‌شوند، مثلاً «اگر شناسهٔ کالا را داری از get_product استفاده کن».

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

توضیح‌های پارامترها را از docstringها حذف کنید و تمرین ۲ را تکرار کنید. توضیح ابزارها چقدر اثر دارد؟

</div>

In [ ]:
import copy


def without_descriptions(schema):
    s = copy.deepcopy(schema)
    for prop in s["function"]["parameters"].get("properties", {}).values():
        prop.pop("description", None)
    s["function"]["description"] = ""
    return s


bare = [without_descriptions(tool_schema(t)) for t in STORE_TOOLS]
acc_bare, details_bare = selection_accuracy(bare)
print(f"with descriptions: {accuracy:.0%}   without descriptions: {acc_bare:.0%}")
details.assign(chosen_without=details_bare.chosen)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

بدون توضیح، مدل فقط نام ابزار و نام پارامترها را می‌بیند. برای ابزارهایی با نام روشن مثل `track_order` تفاوت کم است، ولی جایی که چند ابزار شبیه‌اند (`search_products` و `get_product`)، یا قالب ورودی مهم است (`method` باید `standard` یا `express` باشد)، توضیح نقش اصلی را دارد. اگر بدون توضیح هم دقت بالا بود، یعنی نام‌گذاری ابزارها خوب است، که خودش درس مهمی است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

خروجی `search_products` را طوری تغییر دهید که اگر بیش از ۵ نتیجه وجود دارد، فقط تعداد نتایج و ۵ مورد اول را برگرداند و به مدل بگوید جستجو را دقیق‌تر کند. چرا محدود کردن خروجی ابزار مهم است؟

</div>

In [ ]:
from store_tools import _norm, _product_dict, products


def search_products_limited(query: str, max_results: int = 5) -> dict:
    """Search the product catalogue by name or brand. Returns at most `max_results` matches.

    Args:
        query: words to search for, e.g. "سامسونگ".
        max_results: maximum number of products to return (default 5).
    """
    words = [w for w in _norm(query).split() if len(w) > 1]
    matches = [_product_dict(r) for _, r in products.iterrows()
               if not words or any(w in _norm(f"{r['name']} {r['brand']} {r['category']}") for w in words)]
    matches.sort(key=lambda p: p["price_toman"])
    result = {"count": len(matches), "results": matches[:max_results]}
    if len(matches) > max_results:
        result["note"] = (f"{len(matches)} products match; only the {max_results} cheapest are shown. "
                          "Ask the customer for more details (brand, category or budget) to narrow the search.")
    return result


print(json.dumps(search_products_limited("گوشی"), ensure_ascii=False)[:400])
print(run_with_tools("چه کالاهایی دارید؟", [search_products_limited]))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

محدود کردن خروجی ابزار به چند دلیل مهم است:

- **هزینه و پنجرهٔ متن:** خروجی ابزار وارد پرامپت می‌شود. کاتالوگی با هزار کالا می‌تواند ده‌ها هزار توکن باشد.
- **کیفیت:** همان پدیدهٔ Lost in the Middle جلسهٔ ۸. مدل در فهرست طولانی جزئیات وسط را نادیده می‌گیرد.
- **رفتار بهتر:** پیام `note` به مدل می‌گوید به جای ارائهٔ فهرست بلند، از مشتری بپرسد چه می‌خواهد. همان ایدهٔ پرسش تکمیلی جلسهٔ ۱۵.

تعداد کل نتایج (`count`) را هم برگردانید تا مدل بداند فهرست کامل نیست.

</div>